# Aegis Command - Time-Based Contiguous Holdout Fine-Tuning (Colab GPU Pipeline)

This notebook executes the **50+ Epoch GPU Fine-Tuning Pipeline** with strict **Time-Based Contiguous Splits** on Google Colab.

### Temporal Leakage Prevention (Strict Contiguous Splits):
- **Total Certified Frames**: 155 frames (1,948 ground truth bounding boxes)
- **Fold 1**:
  - Train: Frames `0..107` (108 frames, 69.7%)
  - Purged Buffer: Frames `108..112` (5 frames temporal purge to eliminate overlap)
  - Holdout Test: Frames `113..154` (42 frames, 27.1%)
- **Fold 2 (Reverse)**:
  - Holdout Test: Frames `0..41` (42 frames, 27.1%)
  - Purged Buffer: Frames `42..46` (5 frames temporal purge)
  - Train: Frames `47..154` (108 frames, 69.7%)

### Evaluation Rigor:
- Report metrics for **Fold 1**, **Fold 2**, and **Mean across folds**
- Primary Metric: `person` AP50 with 95% Bootstrap Confidence Intervals (1,000 resamples)
- Dual confidence thresholds: `0.25` and `0.35`
- Direct comparison against:
  1. Matched Pretrained Baseline (zero-shot of same architecture & size)
  2. Best Zero-Shot Bake-Off setting (from Task 2)
- Honest determination: Does domain-specific fine-tuning beat zero-shot scaling?

In [ ]:
# 1. Check GPU allocation
!nvidia-smi

In [ ]:
# 2. Install dependencies
%pip install -q ultralytics albumentations opencv-python matplotlib pandas numpy tabulate

In [ ]:
# 3. Clone Repository & Checkout Branch
!git clone https://github.com/TanmayJain-dev/Aegis-Command.git
%cd Aegis-Command
!git checkout feat/accuracy-v2
!git pull

In [ ]:
# 4. Cryptographic Certification Gate Validation
from pathlib import Path
from vision.eval.eval_detection import check_verification_gate

marker = check_verification_gate(Path("vision/eval/labels"))
print(f"[GATE PASSED] Cryptographically verified ground truth at {marker}")

In [ ]:
# 5. Train Fold 1 (50 Epochs, GPU)
from ultralytics import YOLO
from vision.eval.split_dataset import split_dataset

# Partition Fold 1
split_dataset(fold=1)

print("Training Fold 1 (yolov8s backbone, 50 epochs, imgsz=1280)...\n")
model_f1 = YOLO('yolov8s.pt')
results_f1 = model_f1.train(
    data='vision/data/aegis_data.yaml',
    epochs=50,
    imgsz=1280,
    batch=16,
    device=0,
    workers=4,
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    box=7.5,
    cls=0.5,
    project='runs/train',
    name='aegis_fold1',
    save=True,
    plots=True
)

In [ ]:
# 6. Train Fold 2 (50 Epochs, GPU)
# Partition Fold 2
split_dataset(fold=2)

print("Training Fold 2 (yolov8s backbone, 50 epochs, imgsz=1280)...\n")
model_f2 = YOLO('yolov8s.pt')
results_f2 = model_f2.train(
    data='vision/data/aegis_data.yaml',
    epochs=50,
    imgsz=1280,
    batch=16,
    device=0,
    workers=4,
    lr0=0.01,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.15,
    box=7.5,
    cls=0.5,
    project='runs/train',
    name='aegis_fold2',
    save=True,
    plots=True
)

In [ ]:
# 7. Evaluate Fine-Tuned vs Matched Baseline on Both Folds
from vision.eval.eval_detection import run_evaluation
import json

# Evaluation Matrix across folds
# Fold 1 Test Set (Frames 113..154)
eval_f1_ft = run_evaluation(
    model_path='runs/train/aegis_fold1/weights/best.pt',
    split='val',
    fold=1,
    imgsz=1280,
    conf=0.25,
    device='cuda:0',
    native_classes=True,
    out_results='docs/results/eval_fold1_finetuned.json'
)

eval_f1_baseline = run_evaluation(
    model_path='yolov8s.pt',
    split='val',
    fold=1,
    imgsz=1280,
    conf=0.25,
    device='cuda:0',
    native_classes=False,
    out_results='docs/results/eval_fold1_baseline.json'
)

# Fold 2 Test Set (Frames 0..41)
eval_f2_ft = run_evaluation(
    model_path='runs/train/aegis_fold2/weights/best.pt',
    split='val',
    fold=2,
    imgsz=1280,
    conf=0.25,
    device='cuda:0',
    native_classes=True,
    out_results='docs/results/eval_fold2_finetuned.json'
)

eval_f2_baseline = run_evaluation(
    model_path='yolov8s.pt',
    split='val',
    fold=2,
    imgsz=1280,
    conf=0.25,
    device='cuda:0',
    native_classes=False,
    out_results='docs/results/eval_fold2_baseline.json'
)

print("\n[DONE] Both folds evaluated against matched zero-shot baselines.")

In [ ]:
# 8. Summary Comparison Table & Mean Aggregation
import pandas as pd
import numpy as np

def extract_summary(name, res):
    p = res['metrics_at_conf_0.25']['person']
    ci = res['bootstrap_ci_95_person']
    return {
        'Setting': name,
        'Person AP50': p['ap50'],
        'AP50 95% CI': f"[{ci['ap50_ci'][0]:.3f}, {ci['ap50_ci'][1]:.3f}]",
        'Person Recall': p['recall'],
        'Recall 95% CI': f"[{ci['recall_ci'][0]:.3f}, {ci['recall_ci'][1]:.3f}]",
        'Person P@0.25': p['precision'],
        'Person R@0.25': p['recall'],
        'FP / frame': p['fp_per_frame']
    }

summary_data = [
    extract_summary('Fold 1: Pretrained Baseline (yolov8s @ 1280px)', eval_f1_baseline),
    extract_summary('Fold 1: Fine-Tuned (yolov8s @ 1280px, 50 ep)', eval_f1_ft),
    extract_summary('Fold 2: Pretrained Baseline (yolov8s @ 1280px)', eval_f2_baseline),
    extract_summary('Fold 2: Fine-Tuned (yolov8s @ 1280px, 50 ep)', eval_f2_ft),
]

# Compute 2-fold mean
mean_baseline_ap50 = np.mean([summary_data[0]['Person AP50'], summary_data[2]['Person AP50']])
mean_ft_ap50 = np.mean([summary_data[1]['Person AP50'], summary_data[3]['Person AP50']])
mean_baseline_rec = np.mean([summary_data[0]['Person Recall'], summary_data[2]['Person Recall']])
mean_ft_rec = np.mean([summary_data[1]['Person Recall'], summary_data[3]['Person Recall']])

summary_data.append({
    'Setting': 'MEAN: Pretrained Baseline',
    'Person AP50': round(mean_baseline_ap50, 4),
    'AP50 95% CI': '-',
    'Person Recall': round(mean_baseline_rec, 4),
    'Recall 95% CI': '-',
    'Person P@0.25': round(np.mean([summary_data[0]['Person P@0.25'], summary_data[2]['Person P@0.25']]), 4),
    'Person R@0.25': round(mean_baseline_rec, 4),
    'FP / frame': round(np.mean([summary_data[0]['FP / frame'], summary_data[2]['FP / frame']]), 4)
})

summary_data.append({
    'Setting': 'MEAN: Fine-Tuned (50 ep)',
    'Person AP50': round(mean_ft_ap50, 4),
    'AP50 95% CI': '-',
    'Person Recall': round(mean_ft_rec, 4),
    'Recall 95% CI': '-',
    'Person P@0.25': round(np.mean([summary_data[1]['Person P@0.25'], summary_data[3]['Person P@0.25']]), 4),
    'Person R@0.25': round(mean_ft_rec, 4),
    'FP / frame': round(np.mean([summary_data[1]['FP / frame'], summary_data[3]['FP / frame']]), 4)
})

df_summary = pd.DataFrame(summary_data)
print("### TIME-BASED HOLDOUT FINE-TUNING RESULTS")
print(df_summary.to_markdown(index=False))

delta_ap50 = (mean_ft_ap50 - mean_baseline_ap50) * 100
print(f"\n2-Fold Mean AP50 Delta: {delta_ap50:+.2f} percentage points")

In [ ]:
# 9. Download Weights and Results
from google.colab import files
import subprocess

!zip -r aegis_trained_weights.zip runs/train/aegis_fold1/weights/best.pt runs/train/aegis_fold2/weights/best.pt docs/results/eval_fold*.json
files.download('aegis_trained_weights.zip')